# Experiment 2: finite-sample model coverage

A true complete source model is drawn from a finite library. Clopper–Pearson intervals with Bonferroni correction determine the admissible library subset. The experiment records model coverage, HCR expected-risk bounds, regret and root actions across calibration sizes. Coverage-conditioned and unconditional exceedances must be reported separately.


## Nominal baseline

The plug-in baseline uses a local dynamic program at each state, including states with zero estimated transition probability. HCR, ambiguity construction and interval coverage are evaluated separately.


## Environment


In [ ]:
import sys
import numpy as np
import pandas as pd
import scipy
import matplotlib

print("Python executable:", sys.executable)
print("Python version:", sys.version)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("SciPy:", scipy.__version__)
print("Matplotlib:", matplotlib.__version__)

## Model and solvers


In [ ]:

from __future__ import annotations

import itertools
from dataclasses import dataclass
from functools import lru_cache
from typing import Dict, Iterable, List, Sequence, Tuple

import numpy as np
from scipy.optimize import linprog


Array = np.ndarray


@dataclass(frozen=True)
class SourceSpec:
    """Nominal binary source specification."""
    alpha: float  # P(E=1 | Y=1)
    beta: float   # P(E=1 | Y=0)
    cost: float
    name: str


def make_channel(alpha: float, beta: float) -> Array:
    """
    Return a binary evidence channel with rows Y={0,1} and columns E={0,1}.
    """
    if not (0.0 <= alpha <= 1.0 and 0.0 <= beta <= 1.0):
        raise ValueError("alpha and beta must lie in [0, 1].")
    return np.array(
        [
            [1.0 - beta, beta],
            [1.0 - alpha, alpha],
        ],
        dtype=float,
    )


def _deduplicate_channels(channels: Array, names: Sequence[str]) -> Tuple[Array, List[str]]:
    seen = set()
    kept_channels: List[Array] = []
    kept_names: List[str] = []

    for channel, name in zip(channels, names):
        key = tuple(np.round(channel.ravel(), 12))
        if key not in seen:
            seen.add(key)
            kept_channels.append(channel)
            kept_names.append(name)

    return np.stack(kept_channels), kept_names


def build_coupled_scenarios(
    sources: Sequence[SourceSpec],
    rho: float,
    eps: float = 1e-9,
) -> Tuple[Array, List[List[Tuple[float, float]]], Array, List[str]]:
    """
    Build a finite, coupled ambiguity set.

    The latent regimes shift all sources coherently. This deliberately creates
    dependence across source models so that the history-consistent and fully
    rectangular formulations need not coincide.

    Returns
    -------
    scenario_channels:
        Shape (n_scenarios, n_sources, 2, 2).
    source_projection_points:
        Per-source projected (alpha, beta) channel points for Rect-ROSE.
    nominal_channels:
        Shape (n_sources, 2, 2).
    scenario_names:
        Names of retained coupled regimes.
    """
    if rho < 0:
        raise ValueError("rho must be nonnegative.")

    regimes = [
        (0.0, 0.0, "nominal"),
        (-rho, +rho, "pessimistic"),
        (+rho, -rho, "optimistic"),
        (-rho, -rho, "low-positive-rate"),
        (+rho, +rho, "high-positive-rate"),
    ]

    raw_scenarios: List[Array] = []
    names: List[str] = []

    for delta_alpha, delta_beta, regime_name in regimes:
        scenario = []
        for source in sources:
            alpha = float(np.clip(source.alpha + delta_alpha, eps, 1.0 - eps))
            beta = float(np.clip(source.beta + delta_beta, eps, 1.0 - eps))
            scenario.append(make_channel(alpha, beta))
        raw_scenarios.append(np.stack(scenario))
        names.append(regime_name)

    scenario_channels, names = _deduplicate_channels(np.stack(raw_scenarios), names)

    source_projection_points: List[List[Tuple[float, float]]] = []
    for source_index in range(len(sources)):
        points: List[Tuple[float, float]] = []
        for scenario_index in range(scenario_channels.shape[0]):
            channel = scenario_channels[scenario_index, source_index]
            point = (float(channel[1, 1]), float(channel[0, 1]))
            if not any(np.allclose(point, existing) for existing in points):
                points.append(point)
        source_projection_points.append(points)

    nominal_channels = np.stack(
        [make_channel(source.alpha, source.beta) for source in sources]
    )

    return scenario_channels, source_projection_points, nominal_channels, names


def stop_value_precise(p_y1: float, abstention_cost: float) -> Tuple[float, str]:
    risks = [p_y1, 1.0 - p_y1, abstention_cost]
    actions = ["answer_0", "answer_1", "abstain"]
    index = int(np.argmin(risks))
    return float(risks[index]), actions[index]


def nominal_dp(
    prior_y1: float,
    channels: Array,
    costs: Array,
    abstention_cost: float,
) -> Tuple[Tuple[float, str, Dict[int, float]], Dict]:
    """Exact nominal dynamic program for binary sources."""

    n_sources = len(costs)
    memo: Dict = {}

    def recurse(omega: Array, remaining: Tuple[int, ...]):
        key = (tuple(float(value) for value in omega), remaining)
        if key in memo:
            return memo[key]

        total = float(omega.sum())
        p_y1 = float(omega[1] / total)
        stopping_value, stopping_action = stop_value_precise(
            p_y1, abstention_cost
        )

        best_value = stopping_value
        best_action = stopping_action
        continuation_values: Dict[int, float] = {}

        for source_index in remaining:
            value = float(costs[source_index])
            next_remaining = tuple(
                index for index in remaining if index != source_index
            )

            for evidence in (0, 1):
                next_omega = omega * channels[source_index, :, evidence]
                evidence_probability = float(next_omega.sum() / total)

                if evidence_probability > 0.0:
                    child_value = recurse(next_omega, next_remaining)[0]
                    value += evidence_probability * child_value

            continuation_values[source_index] = value

            if value < best_value - 1e-12:
                best_value = value
                best_action = f"query_{source_index}"

        memo[key] = (best_value, best_action, continuation_values)
        return memo[key]

    root = recurse(
        np.array([1.0 - prior_y1, prior_y1], dtype=float),
        tuple(range(n_sources)),
    )
    return root, memo


def nominal_dp_from_state(
    omega: Array,
    channels: Array,
    costs: Array,
    abstention_cost: float,
    remaining: Tuple[int, ...],
) -> float:
    """Nominal continuation value from an unnormalized binary belief."""

    memo: Dict = {}

    def recurse(current_omega: Array, current_remaining: Tuple[int, ...]) -> float:
        key = (tuple(float(value) for value in current_omega), current_remaining)
        if key in memo:
            return memo[key]

        total = float(current_omega.sum())
        p_y1 = float(current_omega[1] / total)
        best_value, _ = stop_value_precise(p_y1, abstention_cost)

        for source_index in current_remaining:
            value = float(costs[source_index])
            next_remaining = tuple(
                index
                for index in current_remaining
                if index != source_index
            )

            for evidence in (0, 1):
                next_omega = (
                    current_omega * channels[source_index, :, evidence]
                )
                evidence_probability = float(next_omega.sum() / total)

                if evidence_probability > 0.0:
                    value += evidence_probability * recurse(
                        next_omega, next_remaining
                    )

            best_value = min(best_value, value)

        memo[key] = float(best_value)
        return memo[key]

    return recurse(np.asarray(omega, dtype=float), remaining)


def hcr_dp(
    prior_y1: float,
    scenario_channels: Array,
    costs: Array,
    abstention_cost: float,
) -> Tuple[Tuple[float, str, Dict[int, float], Tuple[float, float, float]], Dict]:
    """
    History-Consistent Rectangular dynamic program over a finite scenario set.
    """

    n_scenarios, n_sources, _, _ = scenario_channels.shape
    memo: Dict = {}

    def likelihood_matrix(history: Tuple[Tuple[int, int], ...]) -> Array:
        likelihood = np.tile(
            np.array([1.0 - prior_y1, prior_y1], dtype=float),
            (n_scenarios, 1),
        )

        for source_index, evidence in history:
            likelihood *= scenario_channels[:, source_index, :, evidence]

        return likelihood

    def recurse(
        history: Tuple[Tuple[int, int], ...],
        remaining: Tuple[int, ...],
    ):
        key = (history, remaining)
        if key in memo:
            return memo[key]

        likelihood = likelihood_matrix(history)
        reach_probability = likelihood.sum(axis=1)
        active = reach_probability > 0.0

        if not np.any(active):
            # Every complete model assigns this history zero probability.
            memo[key] = (0.0, "abstain", {}, (0.0, 0.0, float(abstention_cost)))
            return memo[key]

        posterior_y1 = likelihood[active, 1] / reach_probability[active]

        answer_0_risk = float(np.max(posterior_y1))
        answer_1_risk = float(np.max(1.0 - posterior_y1))
        risks = [answer_0_risk, answer_1_risk, abstention_cost]
        actions = ["answer_0", "answer_1", "abstain"]

        stopping_value = float(min(risks))
        stopping_action = actions[int(np.argmin(risks))]

        best_value = stopping_value
        best_action = stopping_action
        continuation_values: Dict[int, float] = {}

        for source_index in remaining:
            next_remaining = tuple(
                index for index in remaining if index != source_index
            )

            child_values = {
                evidence: recurse(
                    history + ((source_index, evidence),),
                    next_remaining,
                )[0]
                for evidence in (0, 1)
            }

            scenario_expectations = np.zeros(n_scenarios, dtype=float)

            for evidence in (0, 1):
                numerator = (
                    likelihood
                    * scenario_channels[:, source_index, :, evidence]
                ).sum(axis=1)

                evidence_probability = np.divide(
                    numerator,
                    reach_probability,
                    out=np.zeros_like(numerator),
                    where=active,
                )

                scenario_expectations += (
                    evidence_probability * child_values[evidence]
                )

            continuation_value = float(
                costs[source_index]
                + np.max(scenario_expectations[active])
            )

            continuation_values[source_index] = continuation_value

            if continuation_value < best_value - 1e-12:
                best_value = continuation_value
                best_action = f"query_{source_index}"

        memo[key] = (
            best_value,
            best_action,
            continuation_values,
            (answer_0_risk, answer_1_risk, abstention_cost),
        )
        return memo[key]

    root = recurse(tuple(), tuple(range(n_sources)))
    return root, memo


def _deduplicate_beliefs(values: Iterable[float]) -> Tuple[float, ...]:
    return tuple(sorted(set(round(float(value), 12) for value in values)))


def rectangular_dp(
    prior_y1: float,
    source_projection_points: Sequence[Sequence[Tuple[float, float]]],
    costs: Array,
    abstention_cost: float,
) -> Tuple[Tuple[float, str, Dict[int, float], Tuple[float, float, float]], Dict]:
    """
    Fully rectangular outer approximation.

    The state retains the projected belief set but discards dependence between
    the belief and the source model that generated it.
    """

    n_sources = len(costs)
    channel_sets = [
        [make_channel(alpha, beta) for alpha, beta in source_points]
        for source_points in source_projection_points
    ]

    memo: Dict = {}

    def recurse(belief_set: Tuple[float, ...], remaining: Tuple[int, ...]):
        belief_set = _deduplicate_beliefs(belief_set)
        key = (belief_set, remaining)

        if key in memo:
            return memo[key]

        beliefs = np.asarray(belief_set, dtype=float)
        if not len(beliefs):
            memo[key] = (0.0, "abstain", {}, (0.0, 0.0, float(abstention_cost)))
            return memo[key]

        answer_0_risk = float(np.max(beliefs))
        answer_1_risk = float(np.max(1.0 - beliefs))
        risks = [answer_0_risk, answer_1_risk, abstention_cost]
        actions = ["answer_0", "answer_1", "abstain"]

        stopping_value = float(min(risks))
        stopping_action = actions[int(np.argmin(risks))]

        best_value = stopping_value
        best_action = stopping_action
        continuation_values: Dict[int, float] = {}

        for source_index in remaining:
            next_remaining = tuple(
                index for index in remaining if index != source_index
            )

            child_belief_sets: Dict[int, Tuple[float, ...]] = {}

            for evidence in (0, 1):
                posteriors: List[float] = []

                for belief in beliefs:
                    for channel in channel_sets[source_index]:
                        denominator = (
                            (1.0 - belief) * channel[0, evidence]
                            + belief * channel[1, evidence]
                        )

                        if denominator > 0.0:
                            posterior = (
                                belief * channel[1, evidence] / denominator
                            )
                            posteriors.append(float(posterior))

                child_belief_sets[evidence] = _deduplicate_beliefs(posteriors)

            child_values = {
                evidence: recurse(
                    child_belief_sets[evidence], next_remaining
                )[0]
                for evidence in (0, 1)
            }

            worst_expectation = -np.inf

            for belief in beliefs:
                for channel in channel_sets[source_index]:
                    expectation = 0.0

                    for evidence in (0, 1):
                        evidence_probability = (
                            (1.0 - belief) * channel[0, evidence]
                            + belief * channel[1, evidence]
                        )
                        expectation += (
                            evidence_probability * child_values[evidence]
                        )

                    worst_expectation = max(
                        worst_expectation, expectation
                    )

            continuation_value = float(
                costs[source_index] + worst_expectation
            )
            continuation_values[source_index] = continuation_value

            if continuation_value < best_value - 1e-12:
                best_value = continuation_value
                best_action = f"query_{source_index}"

        memo[key] = (
            best_value,
            best_action,
            continuation_values,
            (answer_0_risk, answer_1_risk, abstention_cost),
        )
        return memo[key]

    root = recurse((prior_y1,), tuple(range(n_sources)))
    return root, memo


def static_best_response(
    prior_y1: float,
    scenario_channels: Array,
    costs: Array,
    abstention_cost: float,
    adversary_distribution: Array,
):
    """
    Best deterministic policy against a distribution over fixed source models.

    The recursion returns a per-scenario risk vector. At the root, this vector
    is the expected total loss of the policy under each fixed scenario.
    """

    n_scenarios, n_sources, _, _ = scenario_channels.shape
    memo: Dict = {}

    def likelihood_matrix(history: Tuple[Tuple[int, int], ...]) -> Array:
        likelihood = np.tile(
            np.array([1.0 - prior_y1, prior_y1], dtype=float),
            (n_scenarios, 1),
        )

        for source_index, evidence in history:
            likelihood *= scenario_channels[:, source_index, :, evidence]

        return likelihood

    def recurse(
        history: Tuple[Tuple[int, int], ...],
        remaining: Tuple[int, ...],
    ):
        key = (history, remaining)

        if key in memo:
            return memo[key]

        likelihood = likelihood_matrix(history)
        reach_probability = likelihood.sum(axis=1)

        candidates = [
            ("answer_0", likelihood[:, 1].copy(), None),
            ("answer_1", likelihood[:, 0].copy(), None),
            (
                "abstain",
                abstention_cost * reach_probability.copy(),
                None,
            ),
        ]

        for source_index in remaining:
            next_remaining = tuple(
                index for index in remaining if index != source_index
            )
            child_zero = recurse(
                history + ((source_index, 0),), next_remaining
            )
            child_one = recurse(
                history + ((source_index, 1),), next_remaining
            )

            risk_vector = (
                costs[source_index] * reach_probability
                + child_zero[1]
                + child_one[1]
            )

            candidates.append(
                (
                    f"query_{source_index}",
                    risk_vector,
                    (child_zero[2], child_one[2]),
                )
            )

        weighted_values = [
            float(np.dot(adversary_distribution, candidate[1]))
            for candidate in candidates
        ]

        selected_index = int(np.argmin(weighted_values))
        action, risk_vector, children = candidates[selected_index]
        policy_tree = (action, children)

        memo[key] = (
            weighted_values[selected_index],
            risk_vector,
            policy_tree,
        )
        return memo[key]

    root = recurse(tuple(), tuple(range(n_sources)))
    return root, memo


def _solve_restricted_adversary(risk_matrix: Array):
    """
    Solve max_mu min_p <risk_p, mu> for the current policy columns.
    """

    n_policies, n_scenarios = risk_matrix.shape

    # Variables: mu_1,...,mu_S,v. Minimize -v.
    objective = np.zeros(n_scenarios + 1)
    objective[-1] = -1.0

    inequality_matrix = np.zeros((n_policies, n_scenarios + 1))
    inequality_matrix[:, :n_scenarios] = -risk_matrix
    inequality_matrix[:, -1] = 1.0
    inequality_rhs = np.zeros(n_policies)

    equality_matrix = np.zeros((1, n_scenarios + 1))
    equality_matrix[0, :n_scenarios] = 1.0
    equality_rhs = np.array([1.0])

    bounds = [(0.0, None)] * n_scenarios + [(None, None)]

    result = linprog(
        objective,
        A_ub=inequality_matrix,
        b_ub=inequality_rhs,
        A_eq=equality_matrix,
        b_eq=equality_rhs,
        bounds=bounds,
        method="highs",
    )

    if not result.success:
        raise RuntimeError(
            f"Restricted static game failed: {result.message}"
        )

    adversary_distribution = result.x[:n_scenarios]
    game_value = float(result.x[-1])

    return game_value, adversary_distribution


def static_double_oracle(
    prior_y1: float,
    scenario_channels: Array,
    costs: Array,
    abstention_cost: float,
    tolerance: float = 1e-9,
    max_iterations: int = 500,
):
    """
    Solve the exact finite-scenario randomized Static-ROSE game.

    A double-oracle method avoids enumerating all deterministic policy trees.
    The minimizer's best response is obtained by dynamic programming.
    """

    n_scenarios = scenario_channels.shape[0]
    adversary_distribution = np.ones(n_scenarios) / n_scenarios

    first_response, _ = static_best_response(
        prior_y1,
        scenario_channels,
        costs,
        abstention_cost,
        adversary_distribution,
    )

    policy_risk_vectors = [first_response[1]]
    policy_trees = [first_response[2]]
    observed_vectors = {tuple(np.round(first_response[1], 12))}
    convergence_history = []

    for iteration in range(max_iterations):
        risk_matrix = np.vstack(policy_risk_vectors)

        upper_bound, adversary_distribution = (
            _solve_restricted_adversary(risk_matrix)
        )

        best_response, _ = static_best_response(
            prior_y1,
            scenario_channels,
            costs,
            abstention_cost,
            adversary_distribution,
        )

        lower_bound = float(
            np.dot(adversary_distribution, best_response[1])
        )
        gap = float(upper_bound - lower_bound)

        convergence_history.append(
            {
                "iteration": iteration,
                "upper_bound": upper_bound,
                "lower_bound": lower_bound,
                "gap": gap,
                "n_policy_columns": len(policy_risk_vectors),
            }
        )

        if gap <= tolerance:
            return (
                upper_bound,
                adversary_distribution,
                np.vstack(policy_risk_vectors),
                policy_trees,
                convergence_history,
            )

        vector_key = tuple(np.round(best_response[1], 12))

        if vector_key in observed_vectors:
            # Numerical stalling at machine precision.
            return (
                upper_bound,
                adversary_distribution,
                np.vstack(policy_risk_vectors),
                policy_trees,
                convergence_history,
            )

        observed_vectors.add(vector_key)
        policy_risk_vectors.append(best_response[1])
        policy_trees.append(best_response[2])

    raise RuntimeError(
        "Static double-oracle solver reached its iteration limit."
    )


def _robust_stop_from_likelihood(
    likelihood: Array,
    abstention_cost: float,
) -> float:
    reach_probability = likelihood.sum(axis=1)
    active = reach_probability > 0.0

    if not np.any(active):
        return 0.0

    posterior_y1 = (
        likelihood[active, 1] / reach_probability[active]
    )

    return float(
        min(
            np.max(posterior_y1),
            np.max(1.0 - posterior_y1),
            abstention_cost,
        )
    )


def root_acquisition_certificates(
    prior_y1: float,
    scenario_channels: Array,
    costs: Array,
    abstention_cost: float,
):
    """
    Compute root-level perfect-information lower bounds and query-then-stop
    upper bounds for all source actions.
    """

    import pandas as pd

    n_scenarios, n_sources, _, _ = scenario_channels.shape
    root_likelihood = np.tile(
        np.array([1.0 - prior_y1, prior_y1], dtype=float),
        (n_scenarios, 1),
    )

    hcr_root, _ = hcr_dp(
        prior_y1, scenario_channels, costs, abstention_cost
    )
    stopping_value = float(min(hcr_root[3]))

    rows = []

    for source_index in range(n_sources):
        remaining = tuple(
            index for index in range(n_sources) if index != source_index
        )

        child_stopping_values = {}
        for evidence in (0, 1):
            next_likelihood = (
                root_likelihood
                * scenario_channels[:, source_index, :, evidence]
            )
            child_stopping_values[evidence] = (
                _robust_stop_from_likelihood(
                    next_likelihood, abstention_cost
                )
            )

        lower_scenario_values = []
        upper_scenario_values = []

        for scenario_index in range(n_scenarios):
            perfect_information_expectation = 0.0
            query_then_stop_expectation = 0.0

            for evidence in (0, 1):
                next_omega = (
                    root_likelihood[scenario_index]
                    * scenario_channels[
                        scenario_index, source_index, :, evidence
                    ]
                )
                evidence_probability = float(next_omega.sum())

                if evidence_probability > 0.0:
                    perfect_information_expectation += (
                        evidence_probability
                        * nominal_dp_from_state(
                            next_omega,
                            scenario_channels[scenario_index],
                            costs,
                            abstention_cost,
                            remaining,
                        )
                    )
                    query_then_stop_expectation += (
                        evidence_probability
                        * child_stopping_values[evidence]
                    )

            lower_scenario_values.append(
                costs[source_index] + perfect_information_expectation
            )
            upper_scenario_values.append(
                costs[source_index] + query_then_stop_expectation
            )

        lower_bound = float(max(lower_scenario_values))
        upper_bound = float(max(upper_scenario_values))
        exact_hcr_value = float(
            hcr_root[2][source_index]
        )

        rows.append(
            {
                "source_index": source_index,
                "LB": lower_bound,
                "C_HCR": exact_hcr_value,
                "UB": upper_bound,
                "stop_value": stopping_value,
                "bound_valid": (
                    lower_bound
                    <= exact_hcr_value + 1e-10
                    and exact_hcr_value
                    <= upper_bound + 1e-10
                ),
                "rejection_certificate": (
                    lower_bound >= stopping_value - 1e-12
                ),
                "no_harm_certificate": (
                    upper_bound < stopping_value - 1e-12
                ),
            }
        )

    return pd.DataFrame(rows)


def posterior_dilation_unit_test() -> Dict[str, float]:
    """
    Reproduce the analytical cost-free retrieval dilation example.
    """

    prior_y1 = 0.9
    abstention_cost = 0.4
    costs = np.array([0.0])

    channel_a = make_channel(alpha=0.9, beta=0.1)
    channel_b = make_channel(alpha=0.1, beta=0.9)

    scenarios = np.stack(
        [
            np.stack([channel_a]),
            np.stack([channel_b]),
        ]
    )

    root, _ = hcr_dp(
        prior_y1, scenarios, costs, abstention_cost
    )

    initial_stopping_value = float(min(root[3]))
    continuation_value = float(root[2][0])
    robust_value_of_evidence = (
        initial_stopping_value - continuation_value
    )

    expected = {
        "initial_stopping_value": 0.1,
        "post_retrieval_hcr_value": 0.4,
        "RVoE_HCR": -0.3,
    }

    actual = {
        "initial_stopping_value": initial_stopping_value,
        "post_retrieval_hcr_value": continuation_value,
        "RVoE_HCR": robust_value_of_evidence,
    }

    for key, expected_value in expected.items():
        if not np.isclose(actual[key], expected_value, atol=1e-10):
            raise AssertionError(
                f"Dilation unit test failed for {key}: "
                f"expected {expected_value}, obtained {actual[key]}."
            )

    return actual




## Configuration and experiment functions

`RUN_MODE="diagnostic"` uses 20 repetitions and three calibration sizes; `RUN_MODE="final"` uses 1,000 repetitions and six sizes. The stored default is final. Checkpoint/resume support and PDF/SVG figure exports are included.


In [ ]:

# Experiment 2: Finite-Sample Ambiguity Coverage


from pathlib import Path
from typing import Any
import json
import math
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import beta as beta_distribution



# Configuration


SEED = 20260713

# Start with "diagnostic". After it finishes successfully, change to "final".
RUN_MODE = "final"  # allowed values: "diagnostic", "final"

if RUN_MODE == "diagnostic":
    N_REPLICATIONS = 20
    CALIBRATION_SIZES = [25, 100, 500]
    CHECKPOINT_EVERY = 10
elif RUN_MODE == "final":
    N_REPLICATIONS = 1000
    CALIBRATION_SIZES = [25, 50, 100, 200, 500, 1000]
    CHECKPOINT_EVERY = 25
else:
    raise ValueError("RUN_MODE must be either 'diagnostic' or 'final'.")

N_SOURCES = 3
SCENARIO_LIBRARY_SIZE = 41

# Family-wise confidence level for the complete source model.
CONFIDENCE_LEVEL = 0.95
DELTA = 1.0 - CONFIDENCE_LEVEL

# Numerical tolerance for the risk-certificate check.
RISK_TOL = 1e-9

# Resume an interrupted run from its checkpoint.
RESUME = True

# Original local project directory.
PROJECT_DIR = Path("/Users/arian/ROSE-RAG")

# On Vast.ai or another machine, fall back to the current notebook folder.
if not PROJECT_DIR.exists():
    PROJECT_DIR = Path.cwd()

OUTPUT_DIR = (
    PROJECT_DIR
    / f"rose_rag_finite_sample_{RUN_MODE}_outputs"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("RUN_MODE:", RUN_MODE)
print("N_REPLICATIONS:", N_REPLICATIONS)
print("CALIBRATION_SIZES:", CALIBRATION_SIZES)
print("OUTPUT_DIR:", OUTPUT_DIR)


def save_vector_figure(figure, stem: str) -> None:
    """Save only vector PDF and SVG figures."""
    figure.savefig(
        OUTPUT_DIR / f"{stem}.pdf",
        format="pdf",
        bbox_inches="tight",
    )
    figure.savefig(
        OUTPUT_DIR / f"{stem}.svg",
        format="svg",
        bbox_inches="tight",
    )



# FINITE MODEL LIBRARY


def generate_joint_scenario_library(
    rng: np.random.Generator,
    n_sources: int,
    library_size: int,
    minimum_gap: float = 0.08,
) -> tuple[np.ndarray, list[str]]:
    """
    Generate a finite library of coupled complete source models.

    A shared latent reliability shift and a shared positive-rate shift
    create dependence across source channels within each scenario.
    The true source model is selected from this finite library, so the
    HCR solver is exact over the retained ambiguity scenarios.
    """
    if library_size < 2:
        raise ValueError("library_size must be at least 2.")

    base_alpha = rng.uniform(0.72, 0.90, size=n_sources)
    base_beta = rng.uniform(0.10, 0.28, size=n_sources)

    scenarios: list[np.ndarray] = []
    names: list[str] = []

    # Include the baseline as the first scenario.
    baseline_channels = np.stack(
        [
            make_channel(float(alpha), float(beta))
            for alpha, beta in zip(base_alpha, base_beta)
        ]
    )
    scenarios.append(baseline_channels)
    names.append("scenario_000_baseline")

    attempts = 0

    while len(scenarios) < library_size:
        attempts += 1

        if attempts > 200_000:
            raise RuntimeError(
                "Could not generate the requested scenario library."
            )

        shared_reliability_shift = float(rng.normal(0.0, 0.055))
        shared_rate_shift = float(rng.normal(0.0, 0.035))

        source_alpha_noise = rng.normal(
            0.0, 0.018, size=n_sources
        )
        source_beta_noise = rng.normal(
            0.0, 0.018, size=n_sources
        )

        alpha = (
            base_alpha
            + shared_reliability_shift
            + shared_rate_shift
            + source_alpha_noise
        )
        beta = (
            base_beta
            - shared_reliability_shift
            + shared_rate_shift
            + source_beta_noise
        )

        alpha = np.clip(alpha, 0.02, 0.98)
        beta = np.clip(beta, 0.02, 0.98)

        if np.any(alpha - beta < minimum_gap):
            continue

        candidate = np.stack(
            [
                make_channel(float(a), float(b))
                for a, b in zip(alpha, beta)
            ]
        )

        candidate_key = tuple(
            np.round(candidate.ravel(), 10)
        )
        existing_keys = {
            tuple(np.round(existing.ravel(), 10))
            for existing in scenarios
        }

        if candidate_key in existing_keys:
            continue

        scenarios.append(candidate)
        names.append(
            f"scenario_{len(scenarios) - 1:03d}"
        )

    return np.stack(scenarios), names


def extract_alpha_beta(
    channels: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Extract alpha=P(E=1|Y=1) and beta=P(E=1|Y=0).

    Supports:
    - one complete model: shape (m, 2, 2)
    - model library: shape (M, m, 2, 2)
    """
    channels = np.asarray(channels, dtype=float)

    if channels.ndim == 3:
        alpha = channels[:, 1, 1]
        beta = channels[:, 0, 1]
    elif channels.ndim == 4:
        alpha = channels[:, :, 1, 1]
        beta = channels[:, :, 0, 1]
    else:
        raise ValueError("Unexpected channel-array dimension.")

    return alpha, beta



# NESTED CALIBRATION DATA


def generate_nested_calibration_uniforms(
    rng: np.random.Generator,
    n_sources: int,
    maximum_n_cal: int,
) -> dict[str, np.ndarray]:
    """
    Generate uniform random numbers once per replication.

    Smaller calibration datasets use prefixes of the largest dataset,
    making the sample-size comparison nested within each world.
    """
    maximum_n0 = maximum_n_cal // 2
    maximum_n1 = maximum_n_cal - maximum_n0

    return {
        "u_y0": rng.random((n_sources, maximum_n0)),
        "u_y1": rng.random((n_sources, maximum_n1)),
    }


def calibration_counts(
    true_channels: np.ndarray,
    calibration_uniforms: dict[str, np.ndarray],
    n_cal: int,
) -> dict[str, np.ndarray]:
    """
    Return source-wise success counts for balanced conditional samples.

    n_cal is the total number of labeled calibration examples.
    The sample is split as evenly as possible between Y=0 and Y=1.
    """
    true_alpha, true_beta = extract_alpha_beta(true_channels)

    n0 = n_cal // 2
    n1 = n_cal - n0

    e_y0 = (
        calibration_uniforms["u_y0"][:, :n0]
        < true_beta[:, None]
    )
    e_y1 = (
        calibration_uniforms["u_y1"][:, :n1]
        < true_alpha[:, None]
    )

    return {
        "n0": np.full(len(true_alpha), n0, dtype=int),
        "n1": np.full(len(true_alpha), n1, dtype=int),
        "k_beta": e_y0.sum(axis=1).astype(int),
        "k_alpha": e_y1.sum(axis=1).astype(int),
    }



# EXACT SIMULTANEOUS BINOMIAL INTERVALS


def clopper_pearson_interval(
    successes: int,
    trials: int,
    parameter_alpha: float,
) -> tuple[float, float]:
    """
    Exact two-sided Clopper-Pearson interval.

    parameter_alpha is the error probability assigned to this one
    binomial parameter.
    """
    if trials <= 0:
        raise ValueError("trials must be positive.")

    if not (0.0 < parameter_alpha < 1.0):
        raise ValueError("parameter_alpha must lie in (0,1).")

    if successes < 0 or successes > trials:
        raise ValueError("Invalid success count.")

    if successes == 0:
        lower = 0.0
    else:
        lower = float(
            beta_distribution.ppf(
                parameter_alpha / 2.0,
                successes,
                trials - successes + 1,
            )
        )

    if successes == trials:
        upper = 1.0
    else:
        upper = float(
            beta_distribution.ppf(
                1.0 - parameter_alpha / 2.0,
                successes + 1,
                trials - successes,
            )
        )

    return lower, upper


def build_simultaneous_intervals(
    counts: dict[str, np.ndarray],
    n_sources: int,
    familywise_delta: float,
) -> dict[str, np.ndarray]:
    """
    Build simultaneous exact intervals for all 2*m channel parameters.

    Bonferroni allocates delta/(2m) to each alpha_i or beta_i parameter.
    Since each parameter interval is two-sided Clopper-Pearson, the
    resulting complete-model coverage is at least 1-delta by union bound.
    """
    n_parameters = 2 * n_sources
    parameter_alpha = familywise_delta / n_parameters

    alpha_lower = np.zeros(n_sources)
    alpha_upper = np.zeros(n_sources)
    beta_lower = np.zeros(n_sources)
    beta_upper = np.zeros(n_sources)

    alpha_hat = counts["k_alpha"] / counts["n1"]
    beta_hat = counts["k_beta"] / counts["n0"]

    for source_index in range(n_sources):
        alpha_lower[source_index], alpha_upper[source_index] = (
            clopper_pearson_interval(
                int(counts["k_alpha"][source_index]),
                int(counts["n1"][source_index]),
                parameter_alpha,
            )
        )

        beta_lower[source_index], beta_upper[source_index] = (
            clopper_pearson_interval(
                int(counts["k_beta"][source_index]),
                int(counts["n0"][source_index]),
                parameter_alpha,
            )
        )

    return {
        "alpha_hat": alpha_hat,
        "beta_hat": beta_hat,
        "alpha_lower": alpha_lower,
        "alpha_upper": alpha_upper,
        "beta_lower": beta_lower,
        "beta_upper": beta_upper,
        "parameter_alpha": parameter_alpha,
    }


def retain_library_scenarios(
    scenario_library: np.ndarray,
    intervals: dict[str, np.ndarray],
) -> np.ndarray:
    """Return a Boolean mask for models inside all simultaneous intervals."""
    library_alpha, library_beta = extract_alpha_beta(
        scenario_library
    )

    alpha_inside = (
        (library_alpha >= intervals["alpha_lower"][None, :])
        & (library_alpha <= intervals["alpha_upper"][None, :])
    )

    beta_inside = (
        (library_beta >= intervals["beta_lower"][None, :])
        & (library_beta <= intervals["beta_upper"][None, :])
    )

    return np.all(alpha_inside & beta_inside, axis=1)


def nearest_library_scenario(
    scenario_library: np.ndarray,
    intervals: dict[str, np.ndarray],
) -> int:
    """
    Choose the library scenario closest to the empirical channel.

    This fallback is used only if the finite confidence set is empty.
    The fallback event is explicitly recorded and does not count as
    model coverage.
    """
    library_alpha, library_beta = extract_alpha_beta(
        scenario_library
    )

    squared_distance = np.sum(
        (library_alpha - intervals["alpha_hat"][None, :]) ** 2
        + (library_beta - intervals["beta_hat"][None, :]) ** 2,
        axis=1,
    )

    return int(np.argmin(squared_distance))



# POLICY EVALUATION UNDER THE TRUE MODEL


def evaluate_hcr_policy_under_true_model(
    prior_y1: float,
    true_channels: np.ndarray,
    costs: np.ndarray,
    abstention_cost: float,
    hcr_memo: dict,
) -> float:
    """
    Evaluate the HCR policy exactly under one fixed true channel model.
    """
    n_sources = len(costs)

    def recurse(
        history: tuple[tuple[int, int], ...],
        remaining: tuple[int, ...],
        true_omega: np.ndarray,
    ) -> float:
        key = (history, remaining)

        if key not in hcr_memo:
            raise KeyError(
                f"HCR policy state missing for history={history}, "
                f"remaining={remaining}."
            )

        node = hcr_memo[key]
        action = node[1]

        total = float(true_omega.sum())
        true_p_y1 = float(true_omega[1] / total)

        if action == "answer_0":
            return true_p_y1

        if action == "answer_1":
            return 1.0 - true_p_y1

        if action == "abstain":
            return float(abstention_cost)

        if not action.startswith("query_"):
            raise ValueError(f"Unknown HCR action: {action}")

        source_index = int(action.split("_")[1])
        next_remaining = tuple(
            index
            for index in remaining
            if index != source_index
        )

        value = float(costs[source_index])

        for evidence in (0, 1):
            next_true_omega = (
                true_omega
                * true_channels[source_index, :, evidence]
            )
            evidence_probability = float(
                next_true_omega.sum() / total
            )

            if evidence_probability > 0.0:
                value += evidence_probability * recurse(
                    history + ((source_index, evidence),),
                    next_remaining,
                    next_true_omega,
                )

        return float(value)

    initial_omega = np.array(
        [1.0 - prior_y1, prior_y1],
        dtype=float,
    )

    return recurse(
        tuple(),
        tuple(range(n_sources)),
        initial_omega,
    )



def evaluate_nominal_policy_under_true_model(
    prior_y1: float,
    estimated_channels: np.ndarray,
    true_channels: np.ndarray,
    costs: np.ndarray,
    abstention_cost: float,
) -> float:
    """
    Evaluate the plug-in nominal policy under the fixed true channels.

    The nominal action is recomputed from the estimated state using a
    local dynamic program. This avoids reliance on a previously built
    memo whose zero-probability branches may be absent.
    """
    n_sources = len(costs)
    policy_cache: dict = {}

    def estimated_value_and_action(
        estimated_omega: np.ndarray,
        remaining: tuple[int, ...],
    ) -> tuple[float, str]:
        key = (
            tuple(float(value) for value in estimated_omega),
            remaining,
        )

        if key in policy_cache:
            return policy_cache[key]

        total = float(estimated_omega.sum())
        estimated_p_y1 = float(
            estimated_omega[1] / total
        )

        best_value, best_action = stop_value_precise(
            estimated_p_y1,
            abstention_cost,
        )

        for source_index in remaining:
            next_remaining = tuple(
                index
                for index in remaining
                if index != source_index
            )

            continuation_value = float(
                costs[source_index]
            )

            for evidence in (0, 1):
                next_estimated_omega = (
                    estimated_omega
                    * estimated_channels[
                        source_index, :, evidence
                    ]
                )

                evidence_probability = float(
                    next_estimated_omega.sum() / total
                )

                if evidence_probability > 0.0:
                    child_value, _ = (
                        estimated_value_and_action(
                            next_estimated_omega,
                            next_remaining,
                        )
                    )
                    continuation_value += (
                        evidence_probability
                        * child_value
                    )

            if continuation_value < best_value - 1e-12:
                best_value = continuation_value
                best_action = f"query_{source_index}"

        policy_cache[key] = (
            float(best_value),
            str(best_action),
        )
        return policy_cache[key]

    def evaluate_true_risk(
        estimated_omega: np.ndarray,
        true_omega: np.ndarray,
        remaining: tuple[int, ...],
    ) -> float:
        _, action = estimated_value_and_action(
            estimated_omega,
            remaining,
        )

        true_total = float(true_omega.sum())
        true_p_y1 = float(
            true_omega[1] / true_total
        )

        if action == "answer_0":
            return true_p_y1

        if action == "answer_1":
            return 1.0 - true_p_y1

        if action == "abstain":
            return float(abstention_cost)

        source_index = int(action.split("_")[1])
        next_remaining = tuple(
            index
            for index in remaining
            if index != source_index
        )

        value = float(costs[source_index])

        for evidence in (0, 1):
            next_estimated_omega = (
                estimated_omega
                * estimated_channels[
                    source_index, :, evidence
                ]
            )

            next_true_omega = (
                true_omega
                * true_channels[
                    source_index, :, evidence
                ]
            )

            evidence_probability = float(
                next_true_omega.sum() / true_total
            )

            if evidence_probability > 0.0:
                value += evidence_probability * evaluate_true_risk(
                    next_estimated_omega,
                    next_true_omega,
                    next_remaining,
                )

        return float(value)

    initial_omega = np.array(
        [1.0 - prior_y1, prior_y1],
        dtype=float,
    )

    return evaluate_true_risk(
        initial_omega.copy(),
        initial_omega.copy(),
        tuple(range(n_sources)),
    )



def action_class(action: str) -> str:
    if action.startswith("query_"):
        return "query"
    if action == "abstain":
        return "abstain"
    return "answer"



# ONE REPLICATION-WORLD AND ONE SAMPLE SIZE


def build_replication_world(
    replication: int,
) -> dict[str, Any]:
    """
    Build one deterministic world shared across all sample sizes.

    The world and calibration draws depend only on the replication id,
    so checkpoint resume preserves exact reproducibility.
    """
    world_seed = np.random.SeedSequence(
        [SEED, replication, 0]
    )
    data_seed = np.random.SeedSequence(
        [SEED, replication, 1]
    )

    world_rng = np.random.default_rng(world_seed)
    data_rng = np.random.default_rng(data_seed)

    scenario_library, scenario_names = (
        generate_joint_scenario_library(
            world_rng,
            N_SOURCES,
            SCENARIO_LIBRARY_SIZE,
        )
    )

    true_scenario_index = int(
        world_rng.integers(0, SCENARIO_LIBRARY_SIZE)
    )
    true_channels = scenario_library[true_scenario_index]

    prior_y1 = float(world_rng.uniform(0.10, 0.90))
    abstention_cost = float(
        world_rng.uniform(0.20, 0.45)
    )
    costs = world_rng.uniform(
        0.005,
        0.08,
        size=N_SOURCES,
    ).astype(float)

    calibration_uniforms = (
        generate_nested_calibration_uniforms(
            data_rng,
            N_SOURCES,
            max(CALIBRATION_SIZES),
        )
    )

    return {
        "scenario_library": scenario_library,
        "scenario_names": scenario_names,
        "true_scenario_index": true_scenario_index,
        "true_channels": true_channels,
        "prior_y1": prior_y1,
        "abstention_cost": abstention_cost,
        "costs": costs,
        "calibration_uniforms": calibration_uniforms,
    }


def solve_finite_sample_task(
    replication: int,
    n_cal: int,
) -> dict[str, Any]:
    world = build_replication_world(replication)

    scenario_library = world["scenario_library"]
    true_scenario_index = world["true_scenario_index"]
    true_channels = world["true_channels"]
    prior_y1 = world["prior_y1"]
    abstention_cost = world["abstention_cost"]
    costs = world["costs"]

    counts = calibration_counts(
        true_channels,
        world["calibration_uniforms"],
        n_cal,
    )

    intervals = build_simultaneous_intervals(
        counts,
        N_SOURCES,
        DELTA,
    )

    raw_mask = retain_library_scenarios(
        scenario_library,
        intervals,
    )

    true_model_covered = bool(
        raw_mask[true_scenario_index]
    )
    ambiguity_size_raw = int(raw_mask.sum())
    fallback_used = ambiguity_size_raw == 0

    if fallback_used:
        fallback_index = nearest_library_scenario(
            scenario_library,
            intervals,
        )
        retained_indices = np.array(
            [fallback_index],
            dtype=int,
        )
    else:
        retained_indices = np.flatnonzero(raw_mask)

    retained_scenarios = scenario_library[
        retained_indices
    ]

    # Exact HCR solution over the finite retained model set.
    hcr_start = time.perf_counter()
    hcr_root, hcr_memo = hcr_dp(
        prior_y1,
        retained_scenarios,
        costs,
        abstention_cost,
    )
    hcr_runtime = time.perf_counter() - hcr_start

    robust_bound = float(hcr_root[0])
    hcr_action = str(hcr_root[1])

    true_hcr_risk = evaluate_hcr_policy_under_true_model(
        prior_y1,
        true_channels,
        costs,
        abstention_cost,
        hcr_memo,
    )

    risk_violation = bool(
        true_hcr_risk > robust_bound + RISK_TOL
    )

    # Oracle knows the exact true source channels.
    oracle_root, _ = nominal_dp(
        prior_y1,
        true_channels,
        costs,
        abstention_cost,
    )
    oracle_value = float(oracle_root[0])

    # Plug-in nominal policy uses empirical channel estimates.
    # The empirical plug-in estimates can equal exactly 0 or 1 for
    # small calibration samples. A zero-probability estimated branch is
    # absent from the nominal DP memo even though it can occur under the
    # true channel. Clip only the secondary plug-in baseline away from
    # the boundary so every evidence history has a defined action.
    plugin_epsilon = 1e-9

    estimated_channels = np.stack(
        [
            make_channel(
                float(
                    np.clip(
                        intervals["alpha_hat"][i],
                        plugin_epsilon,
                        1.0 - plugin_epsilon,
                    )
                ),
                float(
                    np.clip(
                        intervals["beta_hat"][i],
                        plugin_epsilon,
                        1.0 - plugin_epsilon,
                    )
                ),
            )
            for i in range(N_SOURCES)
        ]
    )

    plugin_root, _ = nominal_dp(
        prior_y1,
        estimated_channels,
        costs,
        abstention_cost,
    )

    plugin_true_risk = (
        evaluate_nominal_policy_under_true_model(
            prior_y1,
            estimated_channels,
            true_channels,
            costs,
            abstention_cost,
        )
    )

    alpha_widths = (
        intervals["alpha_upper"]
        - intervals["alpha_lower"]
    )
    beta_widths = (
        intervals["beta_upper"]
        - intervals["beta_lower"]
    )
    all_widths = np.concatenate(
        [alpha_widths, beta_widths]
    )

    row: dict[str, Any] = {
        "replication": replication,
        "n_cal": n_cal,
        "n0": int(counts["n0"][0]),
        "n1": int(counts["n1"][0]),
        "true_scenario_index": true_scenario_index,
        "prior_y1": prior_y1,
        "abstention_cost": abstention_cost,
        "ambiguity_size_raw": ambiguity_size_raw,
        "ambiguity_size_used": int(len(retained_indices)),
        "fallback_used": fallback_used,
        "true_model_covered": true_model_covered,
        "robust_bound": robust_bound,
        "true_hcr_risk": true_hcr_risk,
        "certificate_slack": (
            robust_bound - true_hcr_risk
        ),
        "risk_violation": risk_violation,
        "risk_violation_when_covered": (
            risk_violation and true_model_covered
        ),
        "oracle_value": oracle_value,
        "oracle_regret_hcr": (
            true_hcr_risk - oracle_value
        ),
        "plugin_true_risk": plugin_true_risk,
        "oracle_regret_plugin": (
            plugin_true_risk - oracle_value
        ),
        "hcr_advantage_vs_plugin": (
            plugin_true_risk - true_hcr_risk
        ),
        "hcr_root_action": hcr_action,
        "hcr_action_class": action_class(hcr_action),
        "plugin_root_action": str(plugin_root[1]),
        "plugin_action_class": action_class(
            str(plugin_root[1])
        ),
        "mean_interval_width": float(
            np.mean(all_widths)
        ),
        "max_interval_width": float(
            np.max(all_widths)
        ),
        "hcr_runtime_sec": hcr_runtime,
        "parameter_alpha": float(
            intervals["parameter_alpha"]
        ),
    }

    true_alpha, true_beta = extract_alpha_beta(
        true_channels
    )

    for source_index in range(N_SOURCES):
        source_number = source_index + 1
        row[f"source_{source_number}_cost"] = float(
            costs[source_index]
        )
        row[f"source_{source_number}_true_alpha"] = float(
            true_alpha[source_index]
        )
        row[f"source_{source_number}_true_beta"] = float(
            true_beta[source_index]
        )
        row[f"source_{source_number}_alpha_hat"] = float(
            intervals["alpha_hat"][source_index]
        )
        row[f"source_{source_number}_beta_hat"] = float(
            intervals["beta_hat"][source_index]
        )
        row[f"source_{source_number}_alpha_lower"] = float(
            intervals["alpha_lower"][source_index]
        )
        row[f"source_{source_number}_alpha_upper"] = float(
            intervals["alpha_upper"][source_index]
        )
        row[f"source_{source_number}_beta_lower"] = float(
            intervals["beta_lower"][source_index]
        )
        row[f"source_{source_number}_beta_upper"] = float(
            intervals["beta_upper"][source_index]
        )

    return row



# SUMMARY STATISTICS


def wilson_interval(
    successes: int,
    trials: int,
    confidence_level: float = 0.95,
) -> tuple[float, float]:
    """Wilson score interval for a binomial proportion."""
    if trials == 0:
        return np.nan, np.nan

    if not 0.0 < confidence_level < 1.0:
        raise ValueError("confidence_level must lie in (0, 1).")
    from scipy.stats import norm
    z = float(norm.ppf((1.0 + confidence_level) / 2.0))
    proportion = successes / trials
    denominator = 1.0 + z**2 / trials

    center = (
        proportion
        + z**2 / (2.0 * trials)
    ) / denominator

    half_width = (
        z
        * math.sqrt(
            proportion * (1.0 - proportion) / trials
            + z**2 / (4.0 * trials**2)
        )
        / denominator
    )

    return max(0.0, center - half_width), min(
        1.0, center + half_width
    )


def summarize_finite_sample_results(
    results_df: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    summary_rows: list[dict[str, Any]] = []
    action_rows: list[dict[str, Any]] = []

    for n_cal, group in results_df.groupby("n_cal"):
        group = group.sort_values("replication")
        n = len(group)

        coverage_count = int(
            group["true_model_covered"].sum()
        )
        violation_count = int(
            group["risk_violation"].sum()
        )

        covered_group = group[
            group["true_model_covered"]
        ]
        conditional_violation_count = int(
            covered_group[
                "risk_violation"
            ].sum()
        )

        coverage_lower, coverage_upper = wilson_interval(
            coverage_count,
            n,
        )
        violation_lower, violation_upper = wilson_interval(
            violation_count,
            n,
        )

        (
            conditional_violation_lower,
            conditional_violation_upper,
        ) = wilson_interval(
            conditional_violation_count,
            len(covered_group),
        )

        summary_rows.append(
            {
                "n_cal": int(n_cal),
                "n_replications": n,
                "channel_coverage": coverage_count / n,
                "channel_coverage_ci_lower": coverage_lower,
                "channel_coverage_ci_upper": coverage_upper,
                "risk_violation_rate": violation_count / n,
                "risk_violation_ci_lower": violation_lower,
                "risk_violation_ci_upper": violation_upper,
                "covered_replications": len(covered_group),
                "conditional_violation_rate": (
                    conditional_violation_count
                    / len(covered_group)
                    if len(covered_group)
                    else np.nan
                ),
                "conditional_violation_ci_lower": (
                    conditional_violation_lower
                ),
                "conditional_violation_ci_upper": (
                    conditional_violation_upper
                ),
                "fallback_rate": float(
                    group["fallback_used"].mean()
                ),
                "mean_certificate_slack": float(
                    group["certificate_slack"].mean()
                ),
                "median_certificate_slack": float(
                    group["certificate_slack"].median()
                ),
                "certificate_slack_q25": float(
                    group["certificate_slack"].quantile(0.25)
                ),
                "certificate_slack_q75": float(
                    group["certificate_slack"].quantile(0.75)
                ),
                "mean_oracle_regret_hcr": float(
                    group["oracle_regret_hcr"].mean()
                ),
                "median_oracle_regret_hcr": float(
                    group["oracle_regret_hcr"].median()
                ),
                "oracle_regret_hcr_q25": float(
                    group["oracle_regret_hcr"].quantile(0.25)
                ),
                "oracle_regret_hcr_q75": float(
                    group["oracle_regret_hcr"].quantile(0.75)
                ),
                "mean_oracle_regret_plugin": float(
                    group["oracle_regret_plugin"].mean()
                ),
                "mean_hcr_advantage_vs_plugin": float(
                    group["hcr_advantage_vs_plugin"].mean()
                ),
                "mean_ambiguity_size": float(
                    group["ambiguity_size_raw"].mean()
                ),
                "median_ambiguity_size": float(
                    group["ambiguity_size_raw"].median()
                ),
                "ambiguity_size_q25": float(
                    group["ambiguity_size_raw"].quantile(0.25)
                ),
                "ambiguity_size_q75": float(
                    group["ambiguity_size_raw"].quantile(0.75)
                ),
                "mean_interval_width": float(
                    group["mean_interval_width"].mean()
                ),
                "mean_hcr_runtime_sec": float(
                    group["hcr_runtime_sec"].mean()
                ),
            }
        )

        for action_name in ["answer", "query", "abstain"]:
            action_rows.append(
                {
                    "n_cal": int(n_cal),
                    "action": action_name,
                    "rate": float(
                        (
                            group["hcr_action_class"]
                            == action_name
                        ).mean()
                    ),
                }
            )

    summary_df = pd.DataFrame(summary_rows).sort_values(
        "n_cal"
    )
    action_df = pd.DataFrame(action_rows).sort_values(
        ["n_cal", "action"]
    )

    return summary_df, action_df



# VECTOR FIGURES


def _sample_size_axis(
    sample_sizes: list[int],
) -> tuple[np.ndarray, list[str]]:
    positions = np.arange(len(sample_sizes))
    labels = [str(value) for value in sample_sizes]
    return positions, labels


def make_channel_coverage_figure(
    summary_df: pd.DataFrame,
) -> None:
    sample_sizes = summary_df["n_cal"].astype(int).tolist()
    x, labels = _sample_size_axis(sample_sizes)

    values = summary_df["channel_coverage"].to_numpy()
    lower = (
        values
        - summary_df[
            "channel_coverage_ci_lower"
        ].to_numpy()
    )
    upper = (
        summary_df[
            "channel_coverage_ci_upper"
        ].to_numpy()
        - values
    )

    figure = plt.figure(figsize=(8.5, 5.2))
    plt.errorbar(
        x,
        values,
        yerr=np.vstack([lower, upper]),
        marker="o",
        capsize=4,
        label="Empirical complete-model coverage",
    )
    plt.axhline(
        CONFIDENCE_LEVEL,
        linestyle="--",
        label="Target coverage",
    )
    plt.xticks(x, labels)
    plt.ylim(0.0, 1.02)
    plt.xlabel("Total calibration sample size")
    plt.ylabel("Coverage probability")
    plt.title(
        "Finite-sample coverage of the true source model"
    )
    plt.legend()
    plt.tight_layout()
    save_vector_figure(
        figure,
        "figure_channel_coverage_vs_sample_size",
    )
    plt.close(figure)


def make_risk_violation_figure(
    summary_df: pd.DataFrame,
) -> None:
    sample_sizes = summary_df["n_cal"].astype(int).tolist()
    x, labels = _sample_size_axis(sample_sizes)

    values = summary_df["risk_violation_rate"].to_numpy()
    lower = (
        values
        - summary_df[
            "risk_violation_ci_lower"
        ].to_numpy()
    )
    upper = (
        summary_df[
            "risk_violation_ci_upper"
        ].to_numpy()
        - values
    )

    figure = plt.figure(figsize=(8.5, 5.2))
    plt.errorbar(
        x,
        values,
        yerr=np.vstack([lower, upper]),
        marker="o",
        capsize=4,
        label="Empirical certificate violation rate",
    )
    plt.axhline(
        DELTA,
        linestyle="--",
        label="Nominal violation level",
    )
    plt.xticks(x, labels)
    plt.ylim(bottom=0.0)
    plt.xlabel("Total calibration sample size")
    plt.ylabel("Violation probability")
    plt.title(
        "True-risk violations of the HCR robust certificate"
    )
    plt.legend()
    plt.tight_layout()
    save_vector_figure(
        figure,
        "figure_risk_violation_vs_sample_size",
    )
    plt.close(figure)


def make_conservatism_figure(
    summary_df: pd.DataFrame,
) -> None:
    sample_sizes = summary_df["n_cal"].astype(int).tolist()
    x, labels = _sample_size_axis(sample_sizes)

    median = summary_df[
        "median_certificate_slack"
    ].to_numpy()
    q25 = summary_df[
        "certificate_slack_q25"
    ].to_numpy()
    q75 = summary_df[
        "certificate_slack_q75"
    ].to_numpy()

    figure = plt.figure(figsize=(8.5, 5.2))
    plt.plot(
        x,
        median,
        marker="o",
        label="Median certificate slack",
    )
    plt.fill_between(
        x,
        q25,
        q75,
        alpha=0.25,
        label="Interquartile range",
    )
    plt.axhline(0.0, linestyle="--")
    plt.xticks(x, labels)
    plt.xlabel("Total calibration sample size")
    plt.ylabel(
        r"$V^{\mathrm{HCR}} - J(\pi^{\mathrm{HCR}},Q^\star)$"
    )
    plt.title(
        "Conservatism of the finite-sample HCR certificate"
    )
    plt.legend()
    plt.tight_layout()
    save_vector_figure(
        figure,
        "figure_conservatism_vs_sample_size",
    )
    plt.close(figure)


def make_action_rate_figure(
    action_df: pd.DataFrame,
) -> None:
    sample_sizes = sorted(
        action_df["n_cal"].unique().astype(int)
    )
    x, labels = _sample_size_axis(sample_sizes)

    figure = plt.figure(figsize=(8.5, 5.2))

    for action_name in ["answer", "query", "abstain"]:
        group = (
            action_df[
                action_df["action"] == action_name
            ]
            .set_index("n_cal")
            .reindex(sample_sizes)
        )
        plt.plot(
            x,
            group["rate"].to_numpy(),
            marker="o",
            label=action_name.capitalize(),
        )

    plt.xticks(x, labels)
    plt.ylim(0.0, 1.0)
    plt.xlabel("Total calibration sample size")
    plt.ylabel("Root-action rate")
    plt.title(
        "HCR answer, retrieval, and abstention rates"
    )
    plt.legend()
    plt.tight_layout()
    save_vector_figure(
        figure,
        "figure_action_rates_vs_sample_size",
    )
    plt.close(figure)


def make_oracle_regret_figure(
    summary_df: pd.DataFrame,
) -> None:
    sample_sizes = summary_df["n_cal"].astype(int).tolist()
    x, labels = _sample_size_axis(sample_sizes)

    median = summary_df[
        "median_oracle_regret_hcr"
    ].to_numpy()
    q25 = summary_df[
        "oracle_regret_hcr_q25"
    ].to_numpy()
    q75 = summary_df[
        "oracle_regret_hcr_q75"
    ].to_numpy()

    figure = plt.figure(figsize=(8.5, 5.2))
    plt.plot(
        x,
        median,
        marker="o",
        label="Median HCR oracle regret",
    )
    plt.fill_between(
        x,
        q25,
        q75,
        alpha=0.25,
        label="Interquartile range",
    )
    plt.axhline(0.0, linestyle="--")
    plt.xticks(x, labels)
    plt.xlabel("Total calibration sample size")
    plt.ylabel("True-risk gap to oracle")
    plt.title(
        "Oracle regret under finite calibration data"
    )
    plt.legend()
    plt.tight_layout()
    save_vector_figure(
        figure,
        "figure_oracle_regret_vs_sample_size",
    )
    plt.close(figure)


def make_ambiguity_size_figure(
    summary_df: pd.DataFrame,
) -> None:
    sample_sizes = summary_df["n_cal"].astype(int).tolist()
    x, labels = _sample_size_axis(sample_sizes)

    median = summary_df[
        "median_ambiguity_size"
    ].to_numpy()
    q25 = summary_df[
        "ambiguity_size_q25"
    ].to_numpy()
    q75 = summary_df[
        "ambiguity_size_q75"
    ].to_numpy()

    figure = plt.figure(figsize=(8.5, 5.2))
    plt.plot(
        x,
        median,
        marker="o",
        label="Median retained model count",
    )
    plt.fill_between(
        x,
        q25,
        q75,
        alpha=0.25,
        label="Interquartile range",
    )
    plt.xticks(x, labels)
    plt.xlabel("Total calibration sample size")
    plt.ylabel("Number of retained source models")
    plt.title(
        "Contraction of the finite ambiguity set"
    )
    plt.legend()
    plt.tight_layout()
    save_vector_figure(
        figure,
        "figure_ambiguity_size_vs_sample_size",
    )
    plt.close(figure)



# CHECKPOINT, RESUME, AND FINAL EXPORT


def experiment_signature() -> dict[str, Any]:
    return {
        "seed": SEED,
        "run_mode": RUN_MODE,
        "n_replications": N_REPLICATIONS,
        "calibration_sizes": CALIBRATION_SIZES,
        "n_sources": N_SOURCES,
        "scenario_library_size": SCENARIO_LIBRARY_SIZE,
        "confidence_level": CONFIDENCE_LEVEL,
        "risk_tolerance": RISK_TOL,
    }


def validate_or_create_checkpoint_signature() -> None:
    signature_path = (
        OUTPUT_DIR / "checkpoint_config.json"
    )
    current_signature = experiment_signature()

    if signature_path.exists():
        existing_signature = json.loads(
            signature_path.read_text(
                encoding="utf-8"
            )
        )

        if existing_signature != current_signature:
            raise RuntimeError(
                "The existing checkpoint belongs to a different "
                "experiment configuration. Either restore the old "
                "configuration, change RUN_MODE, or remove the output "
                "directory before starting a new run."
            )
    else:
        signature_path.write_text(
            json.dumps(
                current_signature,
                indent=2,
            ),
            encoding="utf-8",
        )


def finalize_finite_sample_results(
    results_df: pd.DataFrame,
) -> dict[str, pd.DataFrame]:
    results_df = results_df.sort_values(
        ["n_cal", "replication"]
    ).reset_index(drop=True)

    summary_df, action_df = (
        summarize_finite_sample_results(results_df)
    )

    # Core result tables.
    results_df.to_csv(
        OUTPUT_DIR
        / "results_finite_sample_all_replications.csv",
        index=False,
    )
    summary_df.to_csv(
        OUTPUT_DIR
        / "results_finite_sample_summary.csv",
        index=False,
    )

    summary_df[
        [
            "n_cal",
            "n_replications",
            "channel_coverage",
            "channel_coverage_ci_lower",
            "channel_coverage_ci_upper",
            "fallback_rate",
        ]
    ].to_csv(
        OUTPUT_DIR / "results_channel_coverage.csv",
        index=False,
    )

    summary_df[
        [
            "n_cal",
            "n_replications",
            "risk_violation_rate",
            "risk_violation_ci_lower",
            "risk_violation_ci_upper",
            "conditional_violation_rate",
            "conditional_violation_ci_lower",
            "conditional_violation_ci_upper",
        ]
    ].to_csv(
        OUTPUT_DIR
        / "results_policy_risk_coverage.csv",
        index=False,
    )

    summary_df[
        [
            "n_cal",
            "mean_certificate_slack",
            "median_certificate_slack",
            "certificate_slack_q25",
            "certificate_slack_q75",
        ]
    ].to_csv(
        OUTPUT_DIR / "results_conservatism.csv",
        index=False,
    )

    action_df.to_csv(
        OUTPUT_DIR / "results_action_rates.csv",
        index=False,
    )

    summary_df[
        [
            "n_cal",
            "mean_oracle_regret_hcr",
            "median_oracle_regret_hcr",
            "oracle_regret_hcr_q25",
            "oracle_regret_hcr_q75",
            "mean_oracle_regret_plugin",
            "mean_hcr_advantage_vs_plugin",
        ]
    ].to_csv(
        OUTPUT_DIR / "results_oracle_regret.csv",
        index=False,
    )

    summary_df[
        [
            "n_cal",
            "mean_ambiguity_size",
            "median_ambiguity_size",
            "ambiguity_size_q25",
            "ambiguity_size_q75",
            "mean_interval_width",
        ]
    ].to_csv(
        OUTPUT_DIR / "results_ambiguity_size.csv",
        index=False,
    )

    # Diagnostic tables for any problematic cases.
    results_df[
        results_df["risk_violation"]
    ].to_csv(
        OUTPUT_DIR / "results_risk_violations.csv",
        index=False,
    )

    results_df[
        results_df["risk_violation_when_covered"]
    ].to_csv(
        OUTPUT_DIR
        / "results_risk_violations_when_covered.csv",
        index=False,
    )

    results_df[
        results_df["fallback_used"]
    ].to_csv(
        OUTPUT_DIR / "results_empty_set_fallbacks.csv",
        index=False,
    )

    # Vector figures.
    make_channel_coverage_figure(summary_df)
    make_risk_violation_figure(summary_df)
    make_conservatism_figure(summary_df)
    make_action_rate_figure(action_df)
    make_oracle_regret_figure(summary_df)
    make_ambiguity_size_figure(summary_df)

    metadata = experiment_signature()
    metadata["completed_rows"] = int(len(results_df))
    metadata["completed_replications"] = int(
        results_df["replication"].nunique()
    )

    (
        OUTPUT_DIR / "experiment_metadata.json"
    ).write_text(
        json.dumps(metadata, indent=2),
        encoding="utf-8",
    )

    print("\nFinite-sample experiment finalized.")
    print(f"Output directory: {OUTPUT_DIR}")
    print()
    print(
        summary_df[
            [
                "n_cal",
                "channel_coverage",
                "risk_violation_rate",
                "conditional_violation_rate",
                "median_certificate_slack",
                "median_oracle_regret_hcr",
                "median_ambiguity_size",
                "fallback_rate",
            ]
        ].to_string(index=False)
    )

    return {
        "results": results_df,
        "summary": summary_df,
        "action_rates": action_df,
    }



def run_finite_sample_experiment(
    resume: bool = RESUME,
) -> dict[str, pd.DataFrame]:
    """
    Run or resume the finite-sample experiment.

    Checkpoints are appended in batches rather than rewriting the full
    checkpoint after every batch. This keeps the 6000-task final run
    efficient and makes interruption recovery reliable.
    """
    validate_or_create_checkpoint_signature()

    checkpoint_path = (
        OUTPUT_DIR
        / "checkpoint_finite_sample_replications.csv"
    )

    if not resume and checkpoint_path.exists():
        checkpoint_path.unlink()

    if (
        resume
        and checkpoint_path.exists()
        and checkpoint_path.stat().st_size > 0
    ):
        checkpoint_df = pd.read_csv(checkpoint_path)

        # Keep only the latest copy if a prior interrupted append created
        # duplicate task rows.
        checkpoint_df = (
            checkpoint_df
            .drop_duplicates(
                subset=["replication", "n_cal"],
                keep="last",
            )
            .sort_values(["replication", "n_cal"])
        )

        completed_keys = {
            (int(row.replication), int(row.n_cal))
            for row in checkpoint_df[
                ["replication", "n_cal"]
            ].itertuples(index=False)
        }

        print(
            f"Resuming from {len(completed_keys)} "
            "completed replication-size tasks."
        )
    else:
        completed_keys = set()

        if checkpoint_path.exists():
            checkpoint_path.unlink()

    tasks = [
        (replication, int(n_cal))
        for replication in range(N_REPLICATIONS)
        for n_cal in CALIBRATION_SIZES
    ]

    pending_tasks = [
        task
        for task in tasks
        if task not in completed_keys
    ]

    total_tasks = len(tasks)
    completed_count = len(completed_keys)
    start_time = time.perf_counter()
    batch_rows: list[dict[str, Any]] = []

    def flush_checkpoint_batch() -> None:
        nonlocal batch_rows

        if not batch_rows:
            return

        batch_df = pd.DataFrame(batch_rows)

        write_header = (
            not checkpoint_path.exists()
            or checkpoint_path.stat().st_size == 0
        )

        batch_df.to_csv(
            checkpoint_path,
            mode="a",
            header=write_header,
            index=False,
        )

        batch_rows = []

    for replication, n_cal in pending_tasks:
        row = solve_finite_sample_task(
            replication,
            n_cal,
        )
        batch_rows.append(row)
        completed_count += 1

        if (
            len(batch_rows) >= CHECKPOINT_EVERY
            or completed_count == total_tasks
        ):
            flush_checkpoint_batch()

            elapsed = time.perf_counter() - start_time
            print(
                f"Completed {completed_count}/{total_tasks} "
                f"tasks (elapsed this session {elapsed:.2f} sec)"
            )

    flush_checkpoint_batch()

    if (
        not checkpoint_path.exists()
        or checkpoint_path.stat().st_size == 0
    ):
        raise RuntimeError(
            "The checkpoint file is missing or empty."
        )

    final_df = pd.read_csv(checkpoint_path)

    final_df = (
        final_df
        .drop_duplicates(
            subset=["replication", "n_cal"],
            keep="last",
        )
        .sort_values(["replication", "n_cal"])
        .reset_index(drop=True)
    )

    # Rewrite once at the end to remove any duplicate rows and keep the
    # checkpoint clean for future resume operations.
    final_df.to_csv(
        checkpoint_path,
        index=False,
    )

    expected_rows = (
        N_REPLICATIONS * len(CALIBRATION_SIZES)
    )

    if len(final_df) != expected_rows:
        raise RuntimeError(
            f"Expected {expected_rows} unique task rows but found "
            f"{len(final_df)}. Check the checkpoint."
        )

    return finalize_finite_sample_results(final_df)



def recover_finite_sample_from_checkpoint() -> dict[str, pd.DataFrame]:
    """Finalize an already completed checkpoint without rerunning tasks."""
    checkpoint_path = (
        OUTPUT_DIR
        / "checkpoint_finite_sample_replications.csv"
    )

    if not checkpoint_path.exists():
        raise FileNotFoundError(
            f"Checkpoint not found: {checkpoint_path}"
        )

    checkpoint_df = pd.read_csv(checkpoint_path)
    print(
        f"Recovered {len(checkpoint_df)} completed tasks."
    )
    return finalize_finite_sample_results(
        checkpoint_df
    )



## Execute or resume


In [ ]:
experiment = run_finite_sample_experiment(
    resume=True
)

## Result summaries


In [ ]:
from IPython.display import display

display(experiment["summary"])
display(experiment["action_rates"])

print("Risk violations:")
display(
    experiment["results"].loc[
        experiment["results"]["risk_violation"]
    ]
)

print("Risk violations while the true model was covered:")
display(
    experiment["results"].loc[
        experiment["results"][
            "risk_violation_when_covered"
        ]
    ]
)

print("Empty-set fallbacks:")
display(
    experiment["results"].loc[
        experiment["results"]["fallback_used"]
    ]
)

## Saved vector figures


In [ ]:
from IPython.display import SVG, display

for filename in [
    "figure_channel_coverage_vs_sample_size.svg",
    "figure_risk_violation_vs_sample_size.svg",
    "figure_conservatism_vs_sample_size.svg",
    "figure_action_rates_vs_sample_size.svg",
    "figure_oracle_regret_vs_sample_size.svg",
    "figure_ambiguity_size_vs_sample_size.svg",
]:
    path = OUTPUT_DIR / filename
    if path.exists():
        print(filename)
        display(SVG(filename=str(path)))

## Recover completed results

`recover_finite_sample_from_checkpoint()` regenerates final summaries and figures from completed saved results without repeating the solver sweep.
